# 第 6 周 · 周一：复杂业务问题的SQL解法——从问题到代码的翻译艺术

> 难度：⭐⭐⭐⭐ | 定位：掌握业务问题→SQL步骤的拆解方法论，学会用CTE链式解决复杂分析需求
> 前置依赖：W5窗口函数/JOIN/CTE(工具技能) | 后续用途：W7-W8 AB实验数据准备、W10综合案例
> 面试地位：大厂SQL面试高级题核心，考察分析思维而非语法记忆

---

## 📘 区块1：核心概念与第一性原理

### 1.1 复杂SQL的本质：数据变换管道

任何一个SQL查询，无论看起来多复杂，本质上都是一条"数据变换管道"。原始数据从表里流出来，经过一层层变换，最终变成你需要的答案。

这个管道的每一节，就是一个CTE。理解这一点，你就理解了为什么"复杂SQL = 3-5个CTE串起来"。

**管道模型的优势**：
- 每一步的输出是下一步的输入，数据流向一目了然
- 任何一步都可以独立验证和调试
- 修改某个步骤不影响其他步骤
- 新同事能快速理解你的SQL逻辑

### 1.2 从第一性原理推导：为什么拆解是唯一解？

人脑的工作记忆容量是有限的(心理学中的"7±2法则")。当你试图一次性写出一个5层嵌套的SQL时，你的大脑需要同时跟踪：
- 最内层的WHERE条件
- 中间层的GROUP BY聚合
- 外层的窗口函数
- 最外层的排序和限制

这至少是4×3=12个认知组块，已经超出了大多数人的工作记忆极限。CTE的解决方案：每个CTE只做一件事，给它一个有意义的名字。写完一个验证一个。这样你每次只需要关注一个简单的子问题。

**认知负荷对比**：
- 嵌套子查询方式：一次性加载12+个认知组块，容易出错
- CTE链式方式：每次只关注1-2个组块，逐步构建

### 1.3 分析型SQL的三种思维模型

**模型1：筛选→聚合→排序**
最基础也最常用的模式。先缩小数据范围，再按维度聚合，最后排序取TopN。
```
WHERE过滤 → GROUP BY聚合 → ORDER BY排序 → LIMIT取Top
```

**模型2：分组→编号→筛选**
用于"每个组内取TopN"类问题。按分组字段分窗，用ROW_NUMBER编号，再筛选编号≤N。
```
PARTITION BY分组 → ROW_NUMBER编号 → WHERE rn <= N筛选
```

**模型3：展开→对比→合并**
用于时间序列对比和留存分析。将数据按时间展开成多列，做前后对比，最后汇总统计。
```
LAG/LEAD展开 → 环比/同比计算 → GROUP BY合并
```

### 1.4 本周定位：从工具到组合

W5学了工具——窗口函数、JOIN、CTE、子查询。就像学木工时先学会了锯子、锤子、螺丝刀。

W6学组合——用这些工具做出成品。你知道什么时候用锯子(窗口函数)，什么时候用螺丝刀(JOIN)，什么时候两个都要用。

> **今天最重要的认知**：复杂SQL不需要一次性写出来。先在纸上画出"数据需要经过哪几步变换"，然后每一步写一个CTE，最后串起来。这不是"笨办法"——这是大厂分析师的标准工作流程。

### 1.5 面试高频考点

**面试官常问**：
- "给你一个业务问题，说说你会怎么写SQL？"
- "为什么要用CTE而不是嵌套子查询？"
- "如何保证复杂SQL的正确性？"
- "PERCENT_RANK和ROW_NUMBER在用户分层中分别怎么用？"

**回答要点**：
- 先拆解为3-5个步骤，再说每个步骤用什么SQL工具
- CTE的优势是可读性和可调试性，可以逐步验证
- 逐CTE验证、小样本测试、手工验算
- PERCENT_RANK用于百分比分层(前10%)，ROW_NUMBER用于取固定数量(前10个)

### 1.6 复杂SQL的认知框架：输入→处理→输出

把每个SQL查询看作一个函数：
- **输入**：哪些表？需要什么字段？时间范围是什么？
- **处理**：经过哪些变换？(过滤→聚合→编号→筛选→连接)
- **输出**：最终结果长什么样？粒度是什么？有哪些列？

这个框架帮助你快速梳理任何业务问题的SQL实现路径。先从输出倒推需要的中间数据，再从输入正推能得到什么中间数据，两者在中间汇合——那就是你的CTE链。

## 🧩 区块2：知识点详细拆解

### 2.1 排名+筛选模式(ROW_NUMBER + WHERE)

**为什么重要**：这是"每个组取TopN"的标准解法，也是面试最高频的模式。

**核心语法**：
```sql
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY group_col ORDER BY value_col DESC) AS rn
  FROM source_table
)
SELECT * FROM ranked WHERE rn <= N;
```

**三种排名函数的选择决策**：
| 函数 | 并列处理 | 输出特点 | 适用场景 |
|------|---------|---------|--------|
| ROW_NUMBER | 强制唯一 | 1,2,3,4... | 严格取N个，控制预算 |
| RANK | 跳过名次 | 1,1,3,4... | 排行榜，并列同名 |
| DENSE_RANK | 连续不断 | 1,1,2,3... | 分层(前10%用NTILE更好) |

**实战陷阱**：
- ORDER BY里忘记DESC会拿到最小的N个而不是最大的
- PARTITION BY选错维度会导致全局排名而非组内排名
- 没有处理并列值逻辑：如果第3名有3个人并列，ROW_NUMBER只取一个
- WHERE条件放错位置：应该在CTE外用WHERE rn <= N，而不是在内层

### 2.2 累计+对比模式(SUM OVER + LAG)

**为什么重要**：时间序列分析的核心模式，日报周报的基石。

**累计值计算**：
```sql
SELECT date, amount,
  SUM(amount) OVER (ORDER BY date) AS cum_amount,
  SUM(amount) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling_7d,
  AVG(amount) OVER (ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS ma_30d
FROM daily_sales;
```

**环比/同比计算**：
```sql
SELECT date, amount,
  LAG(amount, 1) OVER (ORDER BY date) AS prev_amount,
  LAG(amount, 12) OVER (ORDER BY date) AS prev_year_amount,
  ROUND((amount - LAG(amount, 1) OVER (ORDER BY date))
    / NULLIF(LAG(amount, 1) OVER (ORDER BY date), 0) * 100, 1) AS mom_pct
FROM monthly_sales;
```

**关键注意**：
- NULLIF防止除零错误——分母为0时返回NULL而非报错
- LAG的offset要根据数据粒度调整：日数据offset=1是环比，月数据offset=12才是同比
- 窗口帧不指定时默认是RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
- 如果数据有缺失日期，ROWS和RANGE的行为不同：ROWS严格按行数，RANGE按值范围

### 2.3 分组+条件模式(GROUP BY + CASE WHEN)

**为什么重要**：用户分群、漏斗统计、多维交叉分析的基础模式。

**条件聚合示例**：
```sql
SELECT user_id,
  SUM(CASE WHEN category = '数码' THEN amount ELSE 0 END) AS digital_spend,
  SUM(CASE WHEN category = '服装' THEN amount ELSE 0 END) AS clothing_spend,
  COUNT(DISTINCT CASE WHEN amount > 500 THEN order_id END) AS big_order_count,
  SUM(CASE WHEN order_date >= '2026-01-01' THEN amount ELSE 0 END) AS recent_spend
FROM orders
GROUP BY user_id;
```

**分群统计**：
```sql
SELECT
  CASE WHEN total_spend >= 1000 THEN '高价值'
       WHEN total_spend >= 300 THEN '中等'
       ELSE '低价值' END AS tier,
  COUNT(*) AS user_count,
  AVG(total_spend) AS avg_spend
FROM user_stats
GROUP BY tier;
```

**常见错误**：
- CASE WHEN的顺序错误：条件从上到下匹配，第一个满足的就返回
- 忘记ELSE：没有ELSE默认返回NULL，统计COUNT时会漏掉
- COUNT(DISTINCT CASE WHEN ...) vs SUM(CASE WHEN ...)：前者去重计数，后者求和

### 2.4 自关联模式(JOIN自己)

**为什么重要**：留存计算、连续行为识别、层级数据查询的核心方法。

**次日留存SQL**：
```sql
SELECT a.date, COUNT(DISTINCT a.user_id) AS d0_users,
  COUNT(DISTINCT b.user_id) AS d1_users
FROM logins a
LEFT JOIN logins b ON a.user_id = b.user_id
  AND b.date = DATE_ADD(a.date, INTERVAL 1 DAY)
GROUP BY a.date;
```

**自关联注意事项**：
- 必须用LEFT JOIN保证分母(day0用户)完整
- JOIN条件既要user_id匹配也要日期间隔匹配
- 大数据场景下自关联性能差——笛卡尔积风险
- 如果只需要7日留存，用b.date = DATE_ADD(a.date, INTERVAL 7 DAY)
- 可以同时计算多日留存：在SELECT中用多个COUNT(DISTINCT CASE WHEN...)统计

**替代方案**：对于大表，可以用窗口函数LAG替代自关联：
```sql
SELECT user_id, date,
  LAG(date, 1) OVER (PARTITION BY user_id ORDER BY date) AS prev_date,
  DATEDIFF(date, LAG(date, 1) OVER (PARTITION BY user_id ORDER BY date)) AS gap
FROM logins;
```

### 2.5 分位+分层模式(NTILE / PERCENT_RANK)

**为什么重要**：用户分层的高级方法，比固定阈值更灵活。

**NTILE：按行数等分**
```sql
SELECT user_id, total_spend,
  NTILE(10) OVER (ORDER BY total_spend DESC) AS decile
FROM user_stats;
-- 将用户分成10等份，第1份是消费最高的10%
```

**PERCENT_RANK：按百分比排名**
```sql
SELECT user_id, total_spend,
  PERCENT_RANK() OVER (ORDER BY total_spend DESC) AS pct_rank
FROM user_stats;
-- 返回值：0到1之间，0是最高，1是最低
```

**选择策略**：
| 需求 | 推荐函数 | 原因 |
|------|---------|------|
| 固定人数(每组50人) | NTILE(N) | 严格等分 |
| 百分比阈值(前10%) | PERCENT_RANK | 百分比语义 |
| 固定名次(前10名) | ROW_NUMBER | 按名次取 |
| 等级划分(ABCD) | NTILE(4) | 四等分 |

**NTILE的注意**：如果总行数不能被N整除，前面的组会多一行。比如103行分10组，前3组有11行，后7组有10行。

### 2.6 SQL调试技巧——写出正确SQL的系统方法

**为什么重要**：再厉害的分析师也会写错SQL。区别在于能不能快速发现和修复错误。

**技巧1：逐CTE验证法**
写完一个CTE后，单独SELECT它：
```sql
WITH step1 AS (...)
SELECT * FROM step1 LIMIT 10;
```
确认step1的数据正确后，再写step2。

**技巧2：小样本测试法**
在数据集的WHERE条件里加上：
```sql
WHERE user_id IN (1, 2, 3)  -- 只测试3个用户
```
如果逻辑对这3个用户正确，大概率对所有用户都正确。

**技巧3：行数检查法**
每一步之后COUNT行数：
- 步骤1后应该有1000行
- 步骤2 JOIN后应该有1000行(LEFT JOIN不会减少)
- 步骤3 GROUP BY后应该有200行
行数不符合预期 = 那一步有问题。

**技巧4：手工验算法**
挑一个具体用户，用手工/Excel算一遍，与SQL结果对比。
这是最可靠但最费时的方法——通常在怀疑SQL有bug但找不出来时使用。

**技巧5：二分定位法**
如果CTE链很长，从中间开始验证：
```sql
WITH step1 AS (...), step2 AS (...), step3 AS (...)
SELECT * FROM step3 LIMIT 10;  -- 跳过step1,2直接看中间结果
```

### 2.7 CTE链式设计原则

**为什么重要**：好的CTE结构让代码自文档化，坏的CTE结构让代码更难读。

**原则1：一个CTE只做一件事**
- 好：`filtered_orders AS (SELECT * FROM orders WHERE dt >= '2026-01-01')`
- 坏：`step1 AS (SELECT ... FROM orders WHERE ... GROUP BY ... HAVING ... ORDER BY ...)`

**原则2：CTE命名要描述"是什么"而不是"做了什么"**
- 好：`user_monthly_spend`, `top10_users`, `daily_retention`
- 坏：`step1`, `temp`, `x`, `t1`

**原则3：CTE的依赖关系应该是线性的**
```sql
WITH raw_data AS (...),        -- 第1步：获取原始数据
     cleaned_data AS (...),    -- 第2步：清洗(依赖raw_data)
     aggregated AS (...),      -- 第3步：聚合(依赖cleaned_data)
     final AS (...)            -- 第4步：最终结果(依赖aggregated)
SELECT * FROM final;
```

**原则4：CTE数量控制**
- 3-5个CTE：理想范围
- 6-8个CTE：可接受，但开始考虑是否有冗余
- 9+个CTE：需要重新审视是否有更简单的方案

### 2.8 复杂SQL的性能考量

**为什么重要**：能跑出结果不等于能在生产环境跑。

**CTE的物化行为(不同数据库)**：
| 数据库 | 默认行为 | 控制方式 |
|--------|---------|--------|
| MySQL 8.0+ | 不物化，作为子查询展开 | 无法手动控制 |
| PostgreSQL | 不物化 | MATERIALIZED / NOT MATERIALIZED |
| SQL Server | 不物化 | 优化器自动决定 |
| BigQuery | 自动决定 | 无需手动控制 |

**优化建议**：
- 如果一个CTE被多次引用且数据量不大，考虑用临时表替代
- 先过滤再JOIN，永远不要把WHERE放到最后
- 避免在CTE中使用SELECT *，只选需要的列
- 大数据量时，考虑窗口函数是否能替代自连接(窗口函数通常更高效)

## 💻 区块3：动手练习

### 步骤0：数据准备

生成500条订单模拟数据(100个用户，4个品类，1月份31天)

### 步骤1：基础练习——每个品类消费金额前20%用户的GMV占比

验证"二八法则"：前20%用户通常贡献40-60%的GMV

In [ ]:
import sqlite3, pandas as pd, numpy as np
np.random.seed(42)
conn = sqlite3.connect(':memory:')

n=500
df=pd.DataFrame({
    'user_id':np.random.choice(range(1,101),n),
    'category':np.random.choice(['服装','数码','食品','美妆'],n),
    'amount':np.round(np.random.lognormal(4,0.7,n),2),
    'order_date':np.random.choice(pd.date_range('2026-01-01','2026-01-31'),n)
})
df['order_date']=df['order_date'].astype(str)
df.to_sql('orders',conn,index=False)

print("问题：每个品类中，消费金额前20%的用户贡献了该品类多少GMV占比？")
print()

q="""WITH user_cat AS (
  SELECT user_id, category, SUM(amount) AS total
  FROM orders GROUP BY user_id, category
),
ranked AS (
  SELECT *,
    PERCENT_RANK() OVER(PARTITION BY category ORDER BY total DESC) AS pct_rk
  FROM user_cat
),
top20 AS (
  SELECT * FROM ranked WHERE pct_rk <= 0.2
),
cat_total AS (
  SELECT category, SUM(total) AS cat_gmv FROM user_cat GROUP BY category
)
SELECT t.category,
  ROUND(SUM(t.total),0) AS top20_gmv,
  ROUND(c.cat_gmv,0) AS total_gmv,
  ROUND(SUM(t.total)/c.cat_gmv*100,1) AS pct
FROM top20 t JOIN cat_total c ON t.category=c.category
GROUP BY t.category ORDER BY pct DESC"""

print(pd.read_sql(q,conn).to_string())
print("\n这就是'二八法则'的验证——前20%用户往往贡献40-60%的GMV")


### 步骤2：进阶练习——每个品类Top5用户 + 用户消费趋势

2.1 找出每个品类消费金额Top5的用户
2.2 分析整体消费趋势(按周聚合)

In [ ]:
# 步骤2.1：每个品类Top5用户
q_top5 = """WITH user_cat AS (
  SELECT user_id, category, SUM(amount) AS total, COUNT(*) AS orders
  FROM orders GROUP BY user_id, category
),
ranked AS (
  SELECT *, ROW_NUMBER() OVER(PARTITION BY category ORDER BY total DESC) AS rn
  FROM user_cat
)
SELECT category, user_id, ROUND(total,0) AS total_spend, orders, rn
FROM ranked WHERE rn <= 5
ORDER BY category, rn"""
print("每个品类Top5用户：")
print(pd.read_sql(q_top5, conn).to_string())

# 步骤2.2：每周消费趋势
print("\n每周消费趋势：")
q_trend = """SELECT DATE_FORMAT(order_date, '%Y-W%u') AS week,
  COUNT(*) AS orders, COUNT(DISTINCT user_id) AS users,
  ROUND(SUM(amount),0) AS gmv, ROUND(AVG(amount),1) AS avg_order
FROM orders GROUP BY week ORDER BY week"""
print(pd.read_sql(q_trend, conn).to_string())


### 步骤3：挑战练习——用户消费分层与交叉分析

3.1 将用户按总消费分为高/中/低三档
3.2 按品类偏好和消费档位做交叉分析
3.3 分析各层用户的消费行为特征差异

In [ ]:
# 步骤3：挑战练习——用户分层与交叉分析
q_challenge = """WITH user_stats AS (
  SELECT user_id,
    SUM(amount) AS total_spend,
    COUNT(*) AS order_count,
    ROUND(AVG(amount),1) AS avg_order
  FROM orders GROUP BY user_id
),
user_tier AS (
  SELECT *,
    CASE WHEN total_spend >= 1000 THEN '高价值'
         WHEN total_spend >= 300 THEN '中等'
         ELSE '低价值' END AS tier
  FROM user_stats
),
user_fav AS (
  SELECT user_id, category,
    ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY SUM(amount) DESC) AS rn
  FROM orders GROUP BY user_id, category
),
user_top_cat AS (
  SELECT user_id, category AS fav_category
  FROM user_fav WHERE rn = 1
)
SELECT t.tier, tc.fav_category,
  COUNT(DISTINCT t.user_id) AS user_count,
  ROUND(AVG(t.total_spend),0) AS avg_spend,
  ROUND(AVG(t.order_count),1) AS avg_orders
FROM user_tier t
JOIN user_top_cat tc ON t.user_id = tc.user_id
GROUP BY t.tier, tc.fav_category
ORDER BY t.tier, user_count DESC"""
print("用户分层 x 品类偏好 交叉分析：")
print(pd.read_sql(q_challenge, conn).to_string())
print("\n这个交叉表可以告诉运营：'高价值用户最爱买什么品类'——精准营销的依据")

# 额外分析：各层用户的消费行为特征
q_profile = """SELECT
  CASE WHEN SUM(amount) >= 1000 THEN '高价值'
       WHEN SUM(amount) >= 300 THEN '中等'
       ELSE '低价值' END AS tier,
  COUNT(DISTINCT user_id) AS users,
  ROUND(AVG(amount),1) AS avg_order_value,
  COUNT(*) * 1.0 / COUNT(DISTINCT user_id) AS orders_per_user,
  COUNT(DISTINCT category) AS category_diversity
FROM orders
GROUP BY tier
ORDER BY AVG(SUM(amount)) OVER() DESC"""
print("\n各层用户的消费行为特征：")
print(pd.read_sql(q_profile, conn).to_string())


## 📊 区块4：业务场景应用

### 场景1：用户分层与精准营销

**业务需求**：市场部需要针对"高价值但30天未购买"的用户发送召回优惠券。

**分析思路**：
1. 计算每个用户的历史总消费(价值维度)
2. 找出最近一次购买日期(活跃维度)
3. 按价值分层和活跃状态交叉筛选

**SQL实现**：
```sql
WITH user_stats AS (
  SELECT user_id,
    SUM(amount) AS total_spend,
    MAX(order_date) AS last_order_date,
    DATEDIFF('2026-02-01', MAX(order_date)) AS days_since_last
  FROM orders
  WHERE order_date <= '2026-02-01'
  GROUP BY user_id
),
user_segments AS (
  SELECT user_id, total_spend, days_since_last,
    CASE WHEN total_spend >= 1000 THEN '高价值'
         WHEN total_spend >= 300 THEN '中等'
         ELSE '低价值' END AS value_tier,
    CASE WHEN days_since_last <= 7 THEN '活跃'
         WHEN days_since_last <= 30 THEN '沉默'
         ELSE '流失' END AS active_status
  FROM user_stats
)
SELECT * FROM user_segments
WHERE value_tier = '高价值' AND active_status = '流失';
```

**不用CTE的后果**：嵌套子查询难以阅读和调试；无法分步验证每个计算是否正确；修改筛选条件时需要改动多个地方。

### 场景2：商品关联分析(购物篮分析)

**业务需求**：找出"买了A品类的用户，有多少也买了B品类"，为跨品类推荐提供数据支持。

**分析思路**：
1. 计算每个用户购买的品类列表
2. 自关联找出购买了品类对的用户
3. 统计每个品类对的共现次数和比例

**SQL实现**：
```sql
WITH user_categories AS (
  SELECT DISTINCT user_id, category FROM orders
),
category_pairs AS (
  SELECT a.category AS cat1, b.category AS cat2,
    COUNT(DISTINCT a.user_id) AS co_users
  FROM user_categories a
  JOIN user_categories b ON a.user_id = b.user_id AND a.category < b.category
  GROUP BY a.category, b.category
),
category_totals AS (
  SELECT category, COUNT(DISTINCT user_id) AS total_users
  FROM user_categories GROUP BY category
)
SELECT cp.cat1, cp.cat2, cp.co_users,
  ROUND(cp.co_users * 100.0 / ct1.total_users, 1) AS pct_of_cat1
FROM category_pairs cp
JOIN category_totals ct1 ON cp.cat1 = ct1.category
ORDER BY cp.co_users DESC;
```

**商业价值**：如果服装用户有60%也买美妆，可以做跨品类推荐/捆绑销售；如果两个品类共现率极低，说明用户群完全不同，营销策略也应该分开。

### 场景3：异常数据检测

**业务需求**：识别消费金额突然翻倍的异常用户(可能被盗刷或数据异常)。

**分析思路**：
1. 按周聚合用户消费
2. 计算环比变化率
3. 筛选环比增长超过阈值的用户

**SQL实现**：
```sql
WITH weekly_spend AS (
  SELECT user_id,
    DATE_FORMAT(order_date, '%Y-%u') AS week,
    SUM(amount) AS weekly_amount
  FROM orders
  GROUP BY user_id, DATE_FORMAT(order_date, '%Y-%u')
),
with_change AS (
  SELECT user_id, week, weekly_amount,
    LAG(weekly_amount) OVER (PARTITION BY user_id ORDER BY week) AS prev_week_amount
  FROM weekly_spend
)
SELECT user_id, week, weekly_amount, prev_week_amount,
  ROUND((weekly_amount - prev_week_amount)
    / NULLIF(prev_week_amount, 0) * 100, 1) AS change_pct
FROM with_change
WHERE prev_week_amount > 0
  AND weekly_amount > prev_week_amount * 2
ORDER BY change_pct DESC;
```

### 场景4：用户生命周期分析

**业务需求**：分析用户从首次购买到最近购买的生命周期长度分布，为不同生命周期阶段的用户设计差异化运营策略。

**SQL实现**：
```sql
WITH user_lifecycle AS (
  SELECT user_id,
    MIN(order_date) AS first_order,
    MAX(order_date) AS last_order,
    DATEDIFF(MAX(order_date), MIN(order_date)) AS lifecycle_days,
    COUNT(*) AS order_count,
    SUM(amount) AS total_spend
  FROM orders
  GROUP BY user_id
)
SELECT
  CASE WHEN lifecycle_days = 0 THEN '当天流失'
       WHEN lifecycle_days <= 7 THEN '1周内'
       WHEN lifecycle_days <= 30 THEN '1月内'
       WHEN lifecycle_days <= 90 THEN '3月内'
       ELSE '3月以上' END AS lifecycle_segment,
  COUNT(*) AS user_count,
  AVG(order_count) AS avg_orders,
  AVG(total_spend) AS avg_spend
FROM user_lifecycle
GROUP BY lifecycle_segment
ORDER BY MIN(lifecycle_days);
```

**商业洞察**：如果"当天流失"的用户占比超过30%，说明首单体验有严重问题；如果"3月以上"的用户平均消费高，说明留存用户的LTV非常高——值得加大留存投入。

## ✅ 区块5：分级检验清单

### 🟢 初级检验(必须能独立完成的)
- [ ] 能说出复杂SQL拆解的3-5步骤法
- [ ] 能写出"排名+筛选"模式的基本SQL(ROW_NUMBER + WHERE rn <= N)
- [ ] 能写出"累计+对比"模式的基本SQL(SUM OVER + LAG)
- [ ] 能用一个CTE解决简单的TopN问题
- [ ] 能写出"分组+条件"模式的条件聚合SQL

**答案提示**：
- 拆解法：先在纸上画出数据变换步骤，每一步一个CTE
- 排名筛选：ROW_NUMBER() OVER(PARTITION BY col ORDER BY val DESC) AS rn，然后WHERE rn <= N
- 累计对比：SUM OVER ORDER BY用于累计，LAG用于获取前一行
- 简单TopN：WITH ranked AS (SELECT *, ROW_NUMBER() OVER(...) rn FROM t) SELECT * FROM ranked WHERE rn <= N
- 条件聚合：SUM(CASE WHEN condition THEN value ELSE 0 END)

### 🟡 中级检验(能解释给同事听的原理)
- [ ] 能解释PERCENT_RANK和ROW_NUMBER在用户分层中的选择策略
- [ ] 能说出CTE链式设计的三个原则
- [ ] 能分析一个复杂SQL的执行顺序是否正确
- [ ] 能运用逐CTE验证法调试SQL
- [ ] 能区分三种排名函数(RANK/DENSE_RANK/ROW_NUMBER)的输出差异

**答案提示**：
- PERCENT_RANK用于百分比阈值(前10%)，ROW_NUMBER用于固定数量(前10名)
- CTE原则：一个CTE只做一件事、命名描述"是什么"、依赖关系线性
- 检查WHERE位置是否正确(是否导致LEFT JOIN变INNER JOIN)、过滤是否在JOIN之前
- 逐CTE验证：写完一个就SELECT测试，确认行数和数据正确后再写下一个
- ROW_NUMBER强制唯一，RANK并列跳跃，DENSE_RANK并列连续

### 🔴 高级检验(能举一反三的迁移能力)
- [ ] 能设计5个以上CTE链式解决复杂业务问题
- [ ] 能根据数据量选择合适的CTE物化策略
- [ ] 能从业务需求直接推导出完整的SQL步骤图
- [ ] 能优化复杂CTE链的性能(减少重复扫描、合并冗余步骤)
- [ ] 能设计包含排名、累计、分组、自关联的复合SQL

**答案提示**：
- 复杂业务问题→画出数据流图→每个节点一个CTE→串联
- 被多次引用的CTE考虑临时表；MySQL需要手动物化
- 从"最终要什么指标"倒推"需要什么中间数据"再倒推"从哪个原始表开始"
- 合并重复的过滤条件、将非必要的CTE内联、在最早步骤做过滤
- 组合模式：CTE1过滤→CTE2排名→CTE3筛选TopN→CTE4聚合→CTE5对比

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W5D2窗口函数(ROW_NUMBER/RANK/LAG) | 排名筛选和累计对比模式的核心工具 |
| **← 前置依赖** | W5D4 JOIN全家桶 | 自关联模式和多表数据整合需要JOIN |
| **← 前置依赖** | W5D5 CTE与子查询 | CTE是拆解复杂SQL的语法基础 |
| **→ 后续用途** | W7-W8 AB实验的数据准备 | 实验分流、指标计算需要复杂SQL |
| **→ 后续用途** | W10综合案例 | 端到端分析需要组合多种SQL模式 |
| **业务连接** | 用户分层与精准营销 | TopN+条件聚合是用户分群的基础 |
| **业务连接** | 异常检测与监控 | 环比/同比分析用于发现数据突变 |

### 横向连接：与其他分析方法论的关联

**与数据分析框架的连接**：
- 用户分层 → RFM模型 → 复杂SQL实现
- 漏斗分析 → AARRR → CTE链式实现
- 同期群分析 → Cohort → 自关联+窗口函数

**与业务决策的连接**：
- TopN分析 → 重点客户识别 → 销售策略
- 用户分群 → 精准营销 → 运营策略
- 趋势分析 → 业务预测 → 战略决策

**与技术架构的连接**：
- CTE vs 临时表 → 查询优化决策
- 增量计算 vs 全量计算 → 数据架构设计
- 实时分析 vs 离线分析 → 技术选型

## 📝 区块7：当日小结

> **复杂SQL = 问题拆解 + CTE链式组合。五大常用模式：排名筛选(ROW_NUMBER)、累计对比(SUM OVER+LAG)、分组条件(GROUP BY+CASE WHEN)、自关联(JOIN自己)、分位分层(NTILE/PERCENT_RANK)。核心技能不是记住语法，而是能快速判断一个业务问题该用哪种模式组合。**

**三条金句带回家**：
1. 复杂SQL不是写出来的，是"拆"出来的——先在纸上画出数据变换管道，再一行行写CTE
2. 每个CTE只做一件事，用命名告诉读者"这个CTE里装的是什么"
3. 调试SQL的最高效方法：逐CTE验证 + 行数检查 + 手工验算

### 🔖 课后拓展

1. **练习**：设计一个SQL解决"找出每个品类中，过去30天购买了≥2次且平均客单价高于品类均值的用户"

2. **思考**：如果一个分析需求需要10个CTE才能完成，是否应该考虑用Python替代SQL？分界线在哪里？

3. **预习**：明天的用户画像SQL实战，将用到今天学的分组+条件模式

4. **面试准备**：准备用白板讲解"给你一个业务问题，你如何设计SQL"的完整思路流程

---

*03analyst_thinking · Week 6 Day 1 · 2026-05-21*